# Funcionalidades Auxiliares

### Obtención de url para Quirk a partir de un json register

Utilidad de prueba: a partir de un registro de algoritmo definido como variable string (json), genera la URL de Quirk codificada y funcional para "https://algassert.com/quirk", validando antes que el circuito se decodifica correctamente (round-trip).


In [14]:
# ----------------------------------------------------------------------------
# Utilidad de prueba: recibe directamente la URL cruda de Quirk (aunque venga
# mal formateada, con "\" y "\"" de más) y la limpia/formatea para que el
# circuito se vea correctamente en https://algassert.com/quirk.
# ----------------------------------------------------------------------------

import re

from IPython.display import HTML, display

from qtrans.utils.qutils import quirk_circuit_info
from qtrans.utils.url_utils import encode_quirk_url, parse_quirk_url

# URL cruda del circuito (mal formateada a propósito: backslashes y comillas)
url_quirk_cruda = (
    '"https://algassert.com/quirk#circuit={\"cols\":[[1,\"•\",\"•\",\"X\",1,1,1,1,1,1,1,1,1],[1,\"•\",\"X\",1,1,1,1,1,1,1,1,1,1],[1,1,1,1,\"•\",\"•\",\"X\",1,1,1,1,1,1],[1,1,1,1,\"•\",\"X\",1,1,1,1,1,1,1],[1,1,1,1,1,1,1,\"•\",\"•\",\"X\",1,1,1],[1,1,1,1,1,1,1,\"•\",\"X\",1,1,1,1],[1,1,1,1,1,1,1,1,1,1,\"•\",\"•\",\"X\"],[1,1,1,1,1,1,1,1,1,1,\"•\",\"X\",1],[\"•\",1,\"•\",\"X\",1,1,1,1,1,1,1,1,1],[1,1,1,\"•\",1,\"•\",\"X\",1,1,1,1,1,1],[1,1,1,1,1,1,\"•\",1,\"•\",\"X\",1,1,1],[1,1,1,1,1,1,1,1,1,\"•\",1,\"•\",\"X\"],[\"•\",1,\"X\",1,1,1,1,1,1,1,1,1,1],[1,1,1,\"•\",1,\"X\",1,1,1,1,1,1,1],[1,1,1,1,1,1,\"•\",1,\"X\",1,1,1,1],[1,1,1,1,1,1,1,1,1,\"•\",1,\"X\",1],[\"Measure\",1,1,1,1,1,1,1,1,1,1,1,1],[1,\"Measure\",1,1,1,1,1,1,1,1,1,1,1],[1,1,\"Measure\",1,1,1,1,1,1,1,1,1,1],[1,1,1,\"Measure\",1,1,1,1,1,1,1,1,1],[1,1,1,1,\"Measure\",1,1,1,1,1,1,1,1],[1,1,1,1,1,\"Measure\",1,1,1,1,1,1,1],[1,1,1,1,1,1,\"Measure\",1,1,1,1,1,1],[1,1,1,1,1,1,1,\"Measure\",1,1,1,1,1],[1,1,1,1,1,1,1,1,\"Measure\",1,1,1,1],[1,1,1,1,1,1,1,1,1,\"Measure\",1,1,1],[1,1,1,1,1,1,1,1,1,1,\"Measure\",1,1],[1,1,1,1,1,1,1,1,1,1,1,\"Measure\",1],[1,1,1,1,1,1,1,1,1,1,1,1,\"Measure\"]]}"'
)


def limpiar_url_quirk(url):
    """Normaliza una URL cruda: quita comillas externas y backslashes sobrantes."""
    if not isinstance(url, str):
        raise ValueError("La URL debe ser una cadena.")
    s = url.strip()
    # Quitar comillas que envuelven toda la cadena
    if len(s) >= 2 and s[0] == s[-1] and s[0] in ('"', "'"):
        s = s[1:-1]
    # Desescapar \"  \'  y eliminar cualquier backslash sobrante
    s = re.sub(r'\\(["\'])', r'\1', s)
    s = s.replace('\\', '')
    s = s.strip().strip('"').strip("'").strip()
    # Si viene con prefijo de registro ("url": ...), cortar desde https://
    idx = s.find('https://')
    if idx > 0:
        s = s[idx:]
    return s


def formatear_url_quirk(url):
    """Acepta la URL cruda (aunque traiga \\ y \" extra) y la percent-encodea."""
    limpia = limpiar_url_quirk(url)
    if "circuit=" not in limpia:
        raise ValueError("La URL debe contener 'circuit=' con el JSON del circuito.")
    return encode_quirk_url(limpia)


def validar_url_quirk(final_url):
    """Valida round-trip: decodifica la URL y reporta info del circuito."""
    try:
        circuito = parse_quirk_url(final_url)
    except Exception:
        print("Estado      : ERROR (no se pudo parsear el circuito)")
        return False
    if isinstance(circuito, dict) and "cols" in circuito:
        info = quirk_circuit_info(final_url)
        print(f"Estado      : OK ({info['n_qubits']} qubits, {info['n_cols']} columnas)")
        return True
    print("Estado      : ERROR (no se pudo parsear el circuito)")
    return False


def mostrar_url_quirk(url):
    """Limpia/formatea la URL recibida y muestra el enlace 'Abrir en Quirk'."""
    final_url = formatear_url_quirk(url)
    validar_url_quirk(final_url)
    display(HTML('<a href="' + final_url + '" target="_blank">Abrir en Quirk</a>'))
    return final_url


# --- Prueba: recibe directamente la URL cruda del circuito -------------------
mostrar_url_quirk(url_quirk_cruda)


Estado      : OK (13 qubits, 29 columnas)


'https://algassert.com/quirk#circuit=%7B%22cols%22%3A%5B%5B1%2C%22%E2%80%A2%22%2C%22%E2%80%A2%22%2C%22X%22%2C1%2C1%2C1%2C1%2C1%2C1%2C1%2C1%2C1%5D%2C%5B1%2C%22%E2%80%A2%22%2C%22X%22%2C1%2C1%2C1%2C1%2C1%2C1%2C1%2C1%2C1%2C1%5D%2C%5B1%2C1%2C1%2C1%2C%22%E2%80%A2%22%2C%22%E2%80%A2%22%2C%22X%22%2C1%2C1%2C1%2C1%2C1%2C1%5D%2C%5B1%2C1%2C1%2C1%2C%22%E2%80%A2%22%2C%22X%22%2C1%2C1%2C1%2C1%2C1%2C1%2C1%5D%2C%5B1%2C1%2C1%2C1%2C1%2C1%2C1%2C%22%E2%80%A2%22%2C%22%E2%80%A2%22%2C%22X%22%2C1%2C1%2C1%5D%2C%5B1%2C1%2C1%2C1%2C1%2C1%2C1%2C%22%E2%80%A2%22%2C%22X%22%2C1%2C1%2C1%2C1%5D%2C%5B1%2C1%2C1%2C1%2C1%2C1%2C1%2C1%2C1%2C1%2C%22%E2%80%A2%22%2C%22%E2%80%A2%22%2C%22X%22%5D%2C%5B1%2C1%2C1%2C1%2C1%2C1%2C1%2C1%2C1%2C1%2C%22%E2%80%A2%22%2C%22X%22%2C1%5D%2C%5B%22%E2%80%A2%22%2C1%2C%22%E2%80%A2%22%2C%22X%22%2C1%2C1%2C1%2C1%2C1%2C1%2C1%2C1%2C1%5D%2C%5B1%2C1%2C1%2C%22%E2%80%A2%22%2C1%2C%22%E2%80%A2%22%2C%22X%22%2C1%2C1%2C1%2C1%2C1%2C1%5D%2C%5B1%2C1%2C1%2C1%2C1%2C1%2C%22%E2%80%A2%22%2C1%2C%22%E2%80%A2%22%2C%22X%22%2C1%2